# Accuracy, entropy and variation between clustering runs

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/lecture-4/accuracy-entropy-and-clustering.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/lecture-4/accuracy-entropy-and-clustering.ipynb)

ABW lecture companion, originally developed by the ABW teaching team. This edition preserves the original sequence of data inspection, models and interpretation. Predict each result before running the cell.


## Prepare the libraries and original teaching data
Install missing libraries, then load the verified raw fruit table. Corrections happen in the lesson below, after inspecting the observations.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'matplotlib': 'matplotlib', 'pandas': 'pandas', 'numpy': 'numpy', 'seaborn': 'seaborn', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/fruits.csv"), Path("fruits.csv")]
                 if p.is_file()), Path("fruits.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/fruits.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22", "Unexpected local data version"


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from itertools import combinations

In [ ]:
the_markers = {
'1': 'Tri_down marker',
'2': 'Tri_up marker',
'3': 'Tri_left marker',
'4': 'Tri_right marker',
'.': 'Point marker',
',': 'Pixel marker',
'o': 'Circle marker',
'v': 'Triangle_down marker',
'^': 'Triangle_up marker',
'<': 'Triangle_left marker',
'>': 'Triangle_right marker',
's': 'Square marker',
'p': 'Pentagon marker',
'*': 'Star marker',
'h': 'Hexagon1 marker',
'H': 'Hexagon2 marker',
'+': 'Plus marker',
'x': 'X marker',
'D': 'Diamond marker',
'd': 'Thin_diamond marker'}

the_colors = {
'r': 'Red',
'g': 'Green',
'b': 'Blue',
'm': 'Magenta',
'y': 'Yellow',
'c': 'Cyan',
'k': 'Black',
'w': 'White'
}

colors = list(the_colors.keys())
markers = list(the_markers.keys())

In [ ]:
def plot_classifier(ax, estimator, X, y,
                   steps=500, colors=colors, markers=markers, ticks=True):
    if X.shape[1] != 2:
        raise ValueError("X must be 2D")

    targets = np.unique(y)
    categories = { c:i for i,c in enumerate(targets) }
    nof_categories = len(categories)

    x_low = np.min(X[:, 0])
    x_high = np.max(X[:, 0])
    y_low = np.min(X[:, 1])
    y_high = np.max(X[:, 1])

    x_extra = (x_high - x_low) * 0.1
    y_extra = (y_high - y_low) * 0.1

    x_low -= x_extra
    x_high += x_extra
    y_low -= y_extra
    y_high += y_extra

    xx, yy = np.meshgrid(
        np.linspace(x_low, x_high, steps), np.linspace(y_low, y_high, steps)
    )

    vectorized_index = np.vectorize(lambda x : categories[x])

    Z = estimator.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = vectorized_index( Z.reshape(xx.shape) )

    ax.contourf(xx, yy, Z, alpha=0.25,
                levels=[level-.1 for level in range(nof_categories+1)],
                colors=colors)

    for i,t in enumerate(targets):
        c = colors[i]
        m = markers[i]
        idx = y == t
        ax.scatter(X[idx,0], X[idx,1], marker=m, c=c, label=t)

    ax.set_xlim(x_low, x_high)
    ax.set_ylim(y_low, y_high)

    ax.get_xaxis().set_visible(ticks)
    ax.get_yaxis().set_visible(ticks)

    ax.legend()

    return ax

## Reuse the correction from Lecture 3

[Lecture 3](../lecture-3/fruit-data-exploration.ipynb) explains the two documented
entry errors and shows the correction code. Here we import `fix_fruit_outliers`.
It returns a copy and matches only those original erroneous records; a second
call leaves corrected data unchanged. It does not discover new outliers.


In [ ]:
# Reuse the correction explained step by step in Lecture 3.
helper = support_path / 'fruit_utils.py'
expected = 'f53f20b3c7abde9aa8001037d116cf641fefd3110d34a364883b4bb0412f2ba9'
if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
    payload = urlopen('https://raw.githubusercontent.com/gromicho/teaching/7fbd913e0d5aebfacce913022a730821065bc49d/support/fruit_utils.py', timeout=45).read()
    if hashlib.sha256(payload).hexdigest() != expected:
        raise ValueError('Fruit helper checksum mismatch')
    helper.write_bytes(payload)
from fruit_utils import fix_fruit_outliers


In [ ]:
raw_fruits = pd.read_csv(data_path, delimiter=';', decimal=',')
fruits = fix_fruit_outliers(raw_fruits)
pd.testing.assert_frame_equal(fix_fruit_outliers(fruits), fruits)
fruits


# Visualize the data

In [ ]:
sns.scatterplot(x='Length', y='Width', data=fruits)
plt.show()

In [ ]:
sns.scatterplot(x='Length', y='Width', data=fruits, hue='Name')
plt.show()

## Classification trees: depth limit and fitted depth

The errors below are **training errors**. A deeper tree can fit these
observations better without predicting unseen observations better.

Start with depth limits 1 through 5, where the tree changes on this dataset.
`max_depth` is an upper limit, not a promise that a fitted tree reaches it.
Each caption and the summary table therefore report both the requested limit
and the actual fitted depth, together with the number of leaves and mistakes.

**Predict:** must every additional level reduce the number of training mistakes?
After comparing the table, optionally try limits 10 through 14. Explain why
several limits can produce the same fitted tree. Use the training/test lesson
in Lecture 3 to explain why zero training errors do not prove good predictions.


In [ ]:
def ShowTheseMaxDepths(max_depths, X, y, features, criterion='entropy',
                       fontsize=5, steps=200, figsize=(13, 5), file_types=()):
    rows = []
    for max_depth in max_depths:
        classifier = tree.DecisionTreeClassifier(
            criterion=criterion, max_depth=max_depth, random_state=0
        ).fit(X, y)
        mistakes = int(np.count_nonzero(classifier.predict(X) != y))
        actual_depth = classifier.get_depth()
        leaves = classifier.get_n_leaves()
        rows.append({'Depth limit': max_depth, 'Fitted depth': actual_depth,
                     'Leaves': leaves, 'Training mistakes': mistakes})
        fig, (ax_tree, ax_surface) = plt.subplots(1, 2, figsize=figsize)
        tree.plot_tree(classifier, ax=ax_tree, filled=True, fontsize=fontsize,
                       feature_names=features, class_names=classifier.classes_.tolist())
        plot_classifier(ax_surface, classifier, X, y, steps=steps)
        fig.suptitle(f'Depth limit {max_depth}; fitted depth {actual_depth}; '
                     f'{leaves} leaves; {mistakes} training mistakes')
        for extension in file_types:
            fig.savefig(f'max_{max_depth}_depth.{extension}', bbox_inches='tight')
        plt.show()
        plt.close(fig)
    return pd.DataFrame(rows)


In [ ]:
fruits_features = ['Length', 'Width']
X = fruits[fruits_features].to_numpy()
y = fruits['Name'].to_numpy()
depth_comparison = ShowTheseMaxDepths(range(1, 6), X, y, fruits_features)
display(depth_comparison)
assert (depth_comparison['Fitted depth'] <= depth_comparison['Depth limit']).all(),     "A fitted tree cannot exceed its requested depth limit."
assert depth_comparison['Leaves'].nunique() > 1,     "Choose depth limits that demonstrate different fitted trees."


In [ ]:
from sklearn.datasets import load_iris
iris_data = load_iris(as_frame=True)
iris = iris_data.data.copy()
iris.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
iris['species'] = [iris_data.target_names[i] for i in iris_data.target]
palette = 'Set1'
s = 65
fig, axs = plt.subplots( 2, 3, figsize=(13,8) )
for ax,(r,c) in zip(axs.flat, combinations(iris.columns[:4],2)):
    sns.scatterplot( ax=ax,
                    x=r, y=c,
                    data=iris, hue="species", palette=palette, s=s)

fig.suptitle("Iris Dataset")
handles, labels = axs.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right')
for ax in axs.flat[1:]:
    ax.legend().set_visible(False)
plt.show()

In [ ]:
iris_not_setosa = iris[iris.species != 'setosa']
features = ['petal_length', 'petal_width']
sns.scatterplot(
    x=features[0], y=features[1],
    data=iris_not_setosa,
    hue="species", palette="Set1", s=s
)
plt.show()

In [ ]:
X = iris_not_setosa[features].values
y = np.ravel( iris_not_setosa.species.values )

ShowTheseMaxDepths( range(2,5), X, y, features, file_types=[] )

In [ ]:
iris_not_versicolor = iris[iris.species != 'versicolor']
features = ['sepal_length', 'sepal_width']
sns.scatterplot(
    x=features[0], y=features[1],
    data=iris_not_versicolor,
    hue="species", palette="Set1", s=s
)
plt.show()

In [ ]:
X = iris_not_versicolor[features].values
y = np.ravel( iris_not_versicolor.species.values )

ShowTheseMaxDepths( range(2,4), X, y, features, file_types=[] )

## Entropy: follow an actual split in the fruit tree

For class proportions $p_1,\ldots,p_C$, entropy in bits is

$$H(p)=-\sum_{c:p_c>0}p_c\log_2 p_c.$$

A pure node has entropy zero. With $C$ possible classes, the maximum is
$\log_2 C$; dividing by this maximum gives normalized entropy. Keep all four
fruit classes in the count vector, including zeros, when comparing these nodes.
We display both our implementation and SciPy's calculation to check the formula.

The two-class vector `[0.5, 0.5]` below is a small warm-up. Then we use the
actual fruit observations and a depth-two tree. Counts refer to training data.


In [ ]:
# Use the same two fruit measurements and depth-two model as in Lecture 3.
entropy_features = ['Length', 'Width']
entropy_tree = tree.DecisionTreeClassifier(
    criterion='entropy', max_depth=2, random_state=0
).fit(fruits[entropy_features], fruits['Name'])
class_order = entropy_tree.classes_.tolist()
membership = entropy_tree.decision_path(fruits[entropy_features]).toarray().astype(bool)
print('Class order:', class_order)


In [ ]:
from math import log2

def Entropy( p ):
  return sum( [ -p*log2(p) if p > 0 else 0 for p in p ])

def EntropyNormalized( p ):
  return Entropy( p ) / log2(len(p))

from scipy.stats import entropy

In [ ]:
p = [ .5, .5 ]
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

### Start with all 92 fruits

The root contains 23 Apples, 20 Bananas, 25 Grapefruits and 24 Oranges.
Compute the class proportions from the table rather than assuming an older
dataset size. The class order printed above also labels the count vectors below.


In [ ]:
counts = fruits['Name'].value_counts().reindex(class_order, fill_value=0)
count = counts.to_numpy()
display(counts.rename('Root count'))
print('Total observations:', int(sum(count)))


In [ ]:
p = [ c/sum(count) for c in count]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

### Follow the right child of the root

For this dataset, that node contains **47 fruits**, with counts
`[0, 20, 25, 2]` in the order Apple, Banana, Grapefruit, Orange. This is a
subset of the 92 fruits, not a different dataset. Use the fitted tree's node
membership to recover the counts and compare them with its displayed diagram.


In [ ]:
parent_id = entropy_tree.tree_.children_right[0]
node_counts = fruits.loc[membership[:, parent_id], 'Name'].value_counts().reindex(
    class_order, fill_value=0
)
assert int(node_counts.sum()) == int(entropy_tree.tree_.n_node_samples[parent_id]),     "The count vector must describe the selected node of the fitted tree."


In [ ]:
node = node_counts.to_list()
p = [c / sum(node) for c in node]
display(node_counts.rename('Parent count'))


In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

### Weight each child's entropy by its share of the parent

The 47-observation node splits into children of sizes **27** and **20**.
Their class counts are `[0, 0, 25, 2]` and `[0, 20, 0, 0]`, respectively.
The second child is pure, so its entropy is zero. Before running the calculation,
predict whether the average entropy after splitting will be smaller:

$$H_{\mathrm{after}}=\frac{27}{47}H(0,0,25/27,2/27)
                   +\frac{20}{47}H(0,1,0,0).$$

The denominator is **47**, the size of this parent, rather than the full root's
92 observations. Information gain at this node is
$H_{\mathrm{parent}}-H_{\mathrm{after}}$. The code derives all counts and weights
from tree membership, so the calculation remains connected to the fitted model.


In [ ]:
child_ids = [entropy_tree.tree_.children_left[parent_id],
             entropy_tree.tree_.children_right[parent_id]]
split_counts = pd.DataFrame([
    fruits.loc[membership[:, child], 'Name'].value_counts().reindex(class_order, fill_value=0)
    for child in child_ids
], index=['Left child', 'Right child'])
child_sizes = split_counts.sum(axis=1)
child_entropies = [Entropy(row / size)
                   for row, size in zip(split_counts.to_numpy(), child_sizes)]
weights = child_sizes / node_counts.sum()
weighted_entropy = sum(weight * value for weight, value in zip(weights, child_entropies))
display(split_counts)
display(pd.DataFrame({'Observations': child_sizes, 'Weight': weights,
                      'Entropy (bits)': child_entropies}))


In [ ]:
print(f'Weighted child entropy: {weighted_entropy:.6f} bits')


In [ ]:
parent_entropy = Entropy(node_counts / node_counts.sum())
information_gain = parent_entropy - weighted_entropy
assert np.array_equal(split_counts.sum(axis=0).to_numpy(), node_counts.to_numpy()),     "Each parent observation must belong to exactly one of its children."
assert np.isclose(weights.sum(), 1), "The child weights must sum to one."
assert np.isclose(parent_entropy, entropy_tree.tree_.impurity[parent_id]),     "Our entropy must match the selected node's entropy in the fitted tree."
assert information_gain >= -1e-12, "This chosen split should not increase entropy."


In [ ]:
print(f'Parent entropy: {parent_entropy:.6f} bits')
print(f'Information gain at this node: {information_gain:.6f} bits')


## Clustering and repeated runs
The six elbow plots deliberately use different seeds. A fixed sequence of distinct seeds makes the experiment reproducible while preserving variability between runs. `n_init=1` means one initialization per fitted model, so we can see the local optima rather than hiding the variation behind multiple restarts.


In [ ]:
data = fruits[fruits_features].copy()

In [ ]:
K = range(1, 11)
fig,axs = plt.subplots(2,3,figsize=(20,8))
for seed, ax in enumerate(axs.flat):
    inertias = []
    for k in K:
        kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
        kmeans.fit(data)
        inertias.append(kmeans.inertia_)

    ax.plot(K, inertias, marker='o')
    ax.grid(True)
    ax.set_xlabel('Number of Clusters (K)')
    ax.set_ylabel('SSE')
    fig.suptitle('Elbow Method for Optimal K, different runs')
plt.show()

# Let us show that `kmeans` may find solutions of very different quality for the same k due to randomness

In [ ]:
k=7
inertias = dict()
for seed in range(1000):
    kmeans = KMeans(n_clusters=k, n_init=1, random_state=seed)
    kmeans.fit(data)
    inertias[kmeans.inertia_] = kmeans

values = sorted(inertias.keys())
best,worse = values[0],values[-1]


for i in [best,worse]:
    centroids = inertias[i].cluster_centers_
    fruits['cluster'] = inertias[i].labels_
    sns.scatterplot(x='Length', y='Width', data=fruits,palette=palette,hue='cluster')
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200, label='Centroids')
    plt.title(inertias[i].inertia_)
    plt.show()
assert worse > best, "Expected different local optima across the distinct starts"
print({"best_SSE": best, "worst_SSE": worse, "distinct_SSE_values": len(values)})


# Comparing the clusters with the actual fruits

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)
fruits['cluster'] = kmeans.labels_
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
a,b=fruits_features
sns.scatterplot(ax=ax1,x=a,y=b,data=fruits,palette=palette,hue='cluster')
sns.scatterplot(ax=ax2,x=a,y=b,data=fruits,palette=palette,hue='Name')
plt.show()

# Linear regression

In [ ]:
lin_reg = LinearRegression()  # Create linear regression object
x_train = fruits[fruits.cluster==0].Length.values.reshape(-1,1)
y_train = fruits[fruits.cluster==0].Width.values.reshape(-1,1)
lin_reg.fit( x_train, y_train )  # Train the model using the training sets

In [ ]:
model_line = lin_reg.predict(x_train)
plt.scatter(x_train, y_train, color='black')
plt.plot(x_train, model_line, color='blue', linewidth=3)
plt.xticks(())
plt.yticks(())
plt.show()

In [ ]:
# Intercept: the value for y when x=0 for the predicted line, \beta_0 in the formulas
lin_reg.intercept_

In [ ]:
# Coefficient: the slope of the predicted line, \beta_1 in the formulas
lin_reg.coef_

In [ ]:
sns.lmplot(x=a,y=b,data=fruits,palette=palette,hue='cluster')
plt.show()

In [ ]:
import numpy as np
assert np.isclose(Entropy([0.5, 0.5]), 1), "Two equally likely classes have entropy 1 bit."
assert np.isclose(Entropy([0, 1]), 0), "A pure node has entropy zero."
assert np.isclose(Entropy(p), entropy(p, base=2)), "Our formula must agree with SciPy."


## What to take away

- Correct the two documented entry errors; do not repeatedly modify extrema.
- Distinguish a requested depth limit from the depth of the fitted tree.
- Compute entropy from the classes in the node being discussed. Weight each
  child's entropy by its share of that parent.
- Different K-means initializations can reach different local optima. The
  repeated-run experiment above deliberately keeps those differences visible.
- Training fit describes observed data. It does not replace evaluation on new
  observations, as demonstrated in [Lecture 3](../lecture-3/fruit-data-exploration.ipynb).

**Explain:** why is the entropy denominator 47 at the selected node, even though
the dataset has 92 rows? Why can a larger depth limit leave the fitted tree
unchanged? Why should a repeated correction call leave the data unchanged?
